# Generador de Disfluencias en Texto

Este notebook implementa un pipeline para generar disfluencias en textos a partir de archivos `.txt` o `.pdf`. El proceso sigue los siguientes pasos para cada oración:

1. **Seleccionar posiciones candidatas**
2. **Aplicar intensidad** (cuántas modificar)
3. **Aplicar severidad** (cómo modificar)
4. **Generar nueva transcripción**

Las disfluencias implementadas son:
- **Repetición de palabras**
- **Repetición de sílabas**
- **Prolongaciones** (repetición de ciertas consonantes)
- **Bloqueos** (espacios simulando silencios)

Se controla la **severidad** (qué tan fuerte es la disfluencia) y la **intensidad** (qué tan frecuente ocurre).

En este ejemplo, se utiliza el archivo `Lectura 1.txt` ubicado en la carpeta `lecturas_text`.

## Paso 1: Cargar y Preprocesar el Texto

En este paso se carga el archivo de texto y se segmenta en oraciones. Además, se filtran oraciones que no sean aptas para la generación de disfluencias (por ejemplo, aquellas que contienen links o valores irrelevantes).

In [ ]:
import re
import os

file_path = os.path.join('lecturas_text_corregidas', 'Lectura 1.txt')

def limpiar_texto(texto: str) -> str:
    # unir líneas cortadas sin punto previo
    texto = re.sub(r'(?<![.!?])\n+', ' ', texto)
    # convertir saltos restantes en separadores
    texto = re.sub(r'\n+', '\n', texto)
    # eliminar comillas simples, dobles, guiones, signos de interrogación...
    texto = re.sub(r'[\-"“”‘’\'¡!¿?\[\]{}()<>#$&=;:,*~^`|/\\]', '', texto)
    # limpiar espacios
    texto = re.sub(r'\s+', ' ', texto)

    return texto.strip()

def es_oracion_valida(oracion: str) -> bool:
    if re.search(r'https?://|www\.', oracion):
        return False
    if len(oracion.strip().split()) < 3:
        return False
    return True

with open(file_path, 'r', encoding='utf-8') as f:
    texto = f.read()

texto = limpiar_texto(texto)

# separar por puntos + posibles saltos
oraciones = re.split(r'(?<=[.!?])\s+', texto)

oraciones_validas = [o.strip() for o in oraciones if es_oracion_valida(o)]

print(f'Oraciones válidas encontradas: {len(oraciones_validas)}')
oraciones_validas[:13]

Oraciones válidas encontradas: 13


['Bono Familiar Universal Midis creará nuevo bono para quienes logren inscribirse en la plataforma Reniec.',
 'La ministra de Desarrollo e Inclusión Social Ariela Luna informó que solo se ha pagado el 32% del bono rural y el 35% del bono familiar universal.',
 'La ministra de Desarrollo e Inclusión Social Ariela Luna informó que las personas que no puedan acceder al Bono Familiar Universal y logren inscribirse en la plataforma de Reniec recibirán un nuevo bono.',
 'Se va a generar un nuevo bono esto no está cerrado por eso hemos creado una plataforma para que se inscriban las personas señaló la ministra.',
 'Además apuntó que solo se ha pagado el 32% del bono rural y el 35% del bono familiar universal.',
 'Durante sesión virtual de la Comisión de Fiscalización y Contraloría la titular del Midis reconoció que se han cometido errores en el reparto del bono e informó del avance en su entrega.',
 'Hemos reconocido que el padrón general de hogares era a demanda y no estaba preparado para es

## Paso 2: Seleccionar posiciones candidatas para disfluencias

En este paso, para cada oración válida, se seleccionan posiciones candidatas donde se pueden insertar disfluencias. Se evita seleccionar palabras irrelevantes (como signos de puntuación, números, etc.).

In [7]:
import random
from typing import List

def obtener_posiciones_candidatas(oracion: str) -> List[int]:
    palabras = re.findall(r'\b\w+\b', oracion)
    posiciones = []
    for i, palabra in enumerate(palabras):
        # Evitar palabras muy cortas o solo números
        if len(palabra) > 2 and not palabra.isdigit():
            posiciones.append(i)
    return posiciones

# Ejemplo con la primera oración válida
oracion_ejemplo = oraciones_validas[0]
posiciones_candidatas = obtener_posiciones_candidatas(oracion_ejemplo)
print('Oración:', oracion_ejemplo)
print('Posiciones candidatas:', posiciones_candidatas)

Oración: Bono Familiar Universal Midis creará nuevo bono para quienes logren inscribirse en la plataforma Reniec.
Posiciones candidatas: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 13, 14]


## Paso 3: Aplicar intensidad y severidad

En este paso se define la cantidad de disfluencias por oración (**intensidad**) y qué tan notorias serán (**severidad**).
- **Intensidad**: baja (1), media (2), alta (3 o más, aleatorio entre 3 y 5).
- **Severidad**: leve, moderado, severo (ajusta los parámetros de cada tipo de disfluencia).

In [8]:
# Parámetros de severidad para cada tipo de disfluencia
SEVERIDAD_PARAMETROS = {
    'leve':    {'rep_pal': (2,2), 'rep_sil': (2,2), 'prol': 1, 'bloq': (3,5)},
    'moderado':{'rep_pal': (2,3), 'rep_sil': (2,3), 'prol': 2, 'bloq': (6,10)},
    'severo':  {'rep_pal': (3,3), 'rep_sil': (3,5), 'prol': 3, 'bloq': (12,20)}
}

def seleccionar_intensidad(intensidad: str, oracion: str) -> int:
    num_palabras = len(oracion.split())
    if intensidad == 'baja':
        return 1
    elif intensidad == 'media':
        return 2
    elif intensidad == 'alta':
        # Entre 30% y 50% de las palabras, mínimo 3 disfluencias
        min_disf = max(3, int(num_palabras * 0.3))
        max_disf = max(min_disf, int(num_palabras * 0.5))
        return random.randint(min_disf, max_disf) if num_palabras > 3 else 3
    else:
        return 1  # por defecto

# Ejemplo de selección de intensidad y severidad
intensidad = random.choice(['baja', 'media', 'alta'])
severidad = random.choice(['leve', 'moderado', 'severo'])

num_disfluencias = seleccionar_intensidad(intensidad, 'Esta es una oración de ejemplo para probar la función.')
parametros = SEVERIDAD_PARAMETROS[severidad]

print(f'Intensidad: {intensidad} -> {num_disfluencias} disfluencias')
print(f'Severidad: {severidad} -> parámetros: {parametros}')

Intensidad: baja -> 1 disfluencias
Severidad: moderado -> parámetros: {'rep_pal': (2, 3), 'rep_sil': (2, 3), 'prol': 2, 'bloq': (6, 10)}


## Paso 4: Generar disfluencias y nueva transcripción

En este paso se aplican las disfluencias seleccionadas a las posiciones candidatas de la oración, generando una nueva transcripción. Se implementan las siguientes funciones:
- Repetición de palabras
- Repetición de sílabas
- Prolongaciones de consonantes
- Bloqueos (espacios)

In [9]:
import unicodedata

def repetir_palabra(palabra, n):
    return ' '.join([palabra]*n)

def repetir_silaba(palabra, n):
    # Extraer sílabas simples (muy básico, para español)
    silabas = re.findall(r'[bcdfghjklmnñpqrstvwxyz]*[aeiouáéíóúü]+', palabra, re.I)
    if not silabas:
        return palabra
    silaba = random.choice(silabas)
    return palabra.replace(silaba, silaba*n, 1)

def prolongacion(palabra, n):
    # Prolongar consonantes m, n, l, s al inicio
    if palabra[0].lower() in ['m','n','l','s']:
        return palabra[0]*n + palabra
    return palabra

def bloqueo(n):
    # Simula un silencio con espacios, n depende de severidad
    return ' ' * n

def aplicar_disfluencia(oracion, posiciones, tipo, parametros):
    palabras = oracion.split()
    for pos in posiciones:
        if pos >= len(palabras): continue
        palabra = palabras[pos]
        if tipo == 'rep_pal':
            n = random.randint(*parametros['rep_pal'])
            palabras[pos] = repetir_palabra(palabra, n)
        elif tipo == 'rep_sil':
            n = random.randint(*parametros['rep_sil'])
            palabras[pos] = repetir_silaba(palabra, n)
        elif tipo == 'prol':
            n = parametros['prol']
            palabras[pos] = prolongacion(palabra, n)
        elif tipo == 'bloq':
            n = random.randint(*parametros['bloq'])
            palabras[pos] = palabra + bloqueo(n)
    return ' '.join(palabras)

# Ejemplo: aplicar una disfluencia aleatoria a la oración de ejemplo
tipos = ['rep_pal', 'rep_sil', 'prol', 'bloq']
tipo = random.choice(tipos)
num_disf = seleccionar_intensidad(intensidad, oracion_ejemplo)
posiciones = random.sample(posiciones_candidatas, min(num_disf, len(posiciones_candidatas)))
nueva_oracion = aplicar_disfluencia(oracion_ejemplo, posiciones, tipo, parametros)
print('Tipo de disfluencia:', tipo)
print('Oración original:', oracion_ejemplo)
print('Oración con disfluencia:', nueva_oracion)

Tipo de disfluencia: prol
Oración original: Bono Familiar Universal Midis creará nuevo bono para quienes logren inscribirse en la plataforma Reniec.
Oración con disfluencia: Bono Familiar Universal Midis creará nuevo bono para quienes logren inscribirse en la plataforma Reniec.


## Ejemplo de pipeline completo sobre varias oraciones

A continuación, se aplica el pipeline completo a las primeras 5 oraciones válidas del archivo, mostrando la oración original y la transcripción con disfluencias generadas.

In [10]:
import os
import pandas as pd

carpeta = 'nuevos_textos'
archivos = [f for f in os.listdir(carpeta) if f.endswith('.txt')]

registros = []

for archivo in archivos:
    file_path = os.path.join(carpeta, archivo)
    
    with open(file_path, 'r', encoding='utf-8') as f:
        texto = f.read()
    
    texto = limpiar_texto(texto)
    oraciones = re.split(r'(?<=[.!?])\s+', texto)
    oraciones_validas = [o.strip() for o in oraciones if es_oracion_valida(o)]
    
    for i, oracion in enumerate(oraciones_validas):
        intensidad = random.choice(['baja', 'media', 'alta'])
        severidad = random.choice(['leve', 'moderado', 'severo'])
        parametros = SEVERIDAD_PARAMETROS[severidad]
        tipo = random.choice(['rep_pal', 'rep_sil', 'prol', 'bloq'])
        
        posiciones_candidatas = obtener_posiciones_candidatas(oracion)
        if not posiciones_candidatas:
            continue
        
        num_disf = seleccionar_intensidad(intensidad, oracion)
        posiciones = random.sample(posiciones_candidatas, min(num_disf, len(posiciones_candidatas)))
        nueva_oracion = aplicar_disfluencia(oracion, posiciones, tipo, parametros)
        
        registros.append({
            'archivo': archivo,
            'oracion_id': i + 1,
            'oracion_original': oracion,
            'oracion_disfluencia': nueva_oracion,
            'intensidad': intensidad,
            'severidad': severidad,
            'tipo_disfluencia': tipo,
            'num_disfluencias': num_disf
        })

df_general = pd.DataFrame(registros)
print(f'Total oraciones procesadas: {len(df_general)}')
df_general.head()

# Exportar DataFrame general a CSV
df_general.to_csv('disfluencias_general_2.csv', index=False, encoding='utf-8')
print('CSV guardado: disfluencias_general_2.csv')

# Crear carpeta de salida
carpeta_salida = 'lecturas_disfluencias'
os.makedirs(carpeta_salida, exist_ok=True)

# Guardar cada archivo modificado
for archivo, grupo in df_general.groupby('archivo'):
    texto_modificado = '\n'.join(grupo['oracion_disfluencia'].tolist())
    ruta_salida = os.path.join(carpeta_salida, archivo)
    with open(ruta_salida, 'w', encoding='utf-8') as f:
        f.write(texto_modificado)

print(f'Archivos modificados guardados en: {carpeta_salida}/')

Total oraciones procesadas: 667
CSV guardado: disfluencias_general_2.csv
Archivos modificados guardados en: lecturas_disfluencias/
